# Практическая работа #7: Подзапросы и операции над множествами в SQL
## Методический ориентир и сквозное эталонное решение (для преподавателя)

Блокнот реализует **сквозную методику**: все теоретические блоки лекции (скалярные подзапросы в `WHERE`, множественные `IN`/`NOT IN`, корреляция, `EXISTS`/`NOT EXISTS`, вычисления в `SELECT`, производные таблицы в `FROM`, реляционные множества, параметризация `?` и экспорт CSV) представлены **на всех трех уровнях сложности**, масштабируясь по числу связанных таблиц и глубине аналитики:

* **🟢 Уровень Easy (4 таблицы):** `members`, `hardware_pcs`, `game_sessions`, `bar_orders`.  
  Базовые скалярные сравнения, защита от ловушки `NULL` в `NOT IN`, локальная корреляция по ПК, предикаты `EXISTS`/`NOT EXISTS`, расчет сводки клиента в `SELECT`, агрегаты над агрегатами в `FROM`, операции над множествами ПК и бара, параметризация, экспорт `client_activity_report.csv`.
* **🟡 Уровень Medium (5 таблиц):** + `tournaments`.  
  Фильтрация по среднему внутри игры, отбор участников турнирных дисциплин, корреляция по категориям бара, поиск спящих гостей через `NOT EXISTS`, аналитическая витрина «Клиент 360», производные таблицы рентабельности категорий, сквозная хронологическая лента через `UNION ALL`, экспорт `client_360_analytics.csv`.
* **🔴 Уровень Hard (6 таблиц):** + `tournament_entries`.  
  Сравнение призовых с фондами соревнований, турнирные бойцы без призовых мест, глубокая корреляция с объединением зон ПК, клиенты-«триатлонисты» (тройной `EXISTS`), скрытые чемпионы, полный расчет Net LTV гостя, многомерная дисперсия в `FROM`, 4-поточный финансовый аудит через `UNION ALL`, экспорт `executive_customer_equity.csv`.
* **Критерии оценивания** на «5», «4», «3», «2».


In [1]:
import sqlite3
import pandas as pd
import pathlib

# Определяем базовый путь к папке работы #7
base_dir = pathlib.Path(".")
if not (base_dir / "easy").exists():
    base_dir = pathlib.Path(r"d:\GitHub\TexhProg-34\07 - Подзапросы")

print("Рабочая директория:", base_dir.resolve())
print("Версия Pandas:", pd.__version__)
print("Версия SQLite3:", sqlite3.sqlite_version)


Рабочая директория: D:\GitHub\TexhProg-34\07 - Подзапросы
Версия Pandas: 3.0.2
Версия SQLite3: 3.50.4


---
# 🟢 Часть 1. Эталонное решение Уровня Easy (Базовый)
**4 таблицы:** `members`, `hardware_pcs`, `game_sessions`, `bar_orders`

### Задание 1.1–1.6: Подготовка базы данных и миграция CSV (вариант 1)


In [2]:
# Загрузка 4 CSV-файлов варианта 1
easy_dir = base_dir / "easy" / "variant_01"
df_members_e = pd.read_csv(easy_dir / "members.csv", sep=";")
df_pcs_e = pd.read_csv(easy_dir / "hardware_pcs.csv", sep=";")
df_sessions_e = pd.read_csv(easy_dir / "game_sessions.csv", sep=";")
df_bar_e = pd.read_csv(easy_dir / "bar_orders.csv", sep=";")

# Создание базы данных в памяти для демонстрации
conn_easy = sqlite3.connect(":memory:")
conn_easy.execute("PRAGMA foreign_keys = ON;")
cur_e = conn_easy.cursor()

# Создание схемы 4 таблиц
cur_e.executescript('''
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    nickname TEXT NOT NULL,
    city TEXT NOT NULL,
    email TEXT,
    registered_at TEXT NOT NULL,
    loyalty_tier TEXT NOT NULL,
    balance INTEGER NOT NULL
);

CREATE TABLE hardware_pcs (
    pc_id INTEGER PRIMARY KEY,
    pc_number TEXT NOT NULL,
    zone_name TEXT NOT NULL,
    hourly_rate INTEGER NOT NULL,
    gpu TEXT NOT NULL,
    cpu TEXT NOT NULL,
    ram_gb INTEGER NOT NULL,
    status TEXT NOT NULL
);

CREATE TABLE game_sessions (
    session_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES members(member_id),
    pc_id INTEGER REFERENCES hardware_pcs(pc_id),
    game_title TEXT NOT NULL,
    session_date TEXT NOT NULL,
    hours_played REAL NOT NULL,
    discount_promo INTEGER NOT NULL,
    total_paid INTEGER NOT NULL,
    client_rating INTEGER
);

CREATE TABLE bar_orders (
    order_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES members(member_id),
    item_name TEXT NOT NULL,
    category TEXT NOT NULL,
    order_date TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    item_price INTEGER NOT NULL,
    total_amount INTEGER NOT NULL
);
''')

# Импорт данных
df_members_e.to_sql("members", conn_easy, if_exists="append", index=False)
df_pcs_e.to_sql("hardware_pcs", conn_easy, if_exists="append", index=False)
df_sessions_e.to_sql("game_sessions", conn_easy, if_exists="append", index=False)
df_bar_e.to_sql("bar_orders", conn_easy, if_exists="append", index=False)
conn_easy.commit()

# Аудит строк
audit_query = '''
SELECT 'members' AS tbl, COUNT(*) AS sql_count, ? AS pd_len FROM members
UNION ALL
SELECT 'hardware_pcs', COUNT(*), ? FROM hardware_pcs
UNION ALL
SELECT 'game_sessions', COUNT(*), ? FROM game_sessions
UNION ALL
SELECT 'bar_orders', COUNT(*), ? FROM bar_orders;
'''
pd.read_sql_query(audit_query, conn_easy, params=(len(df_members_e), len(df_pcs_e), len(df_sessions_e), len(df_bar_e)))


tbl,sql_count,pd_len
members,100,100
hardware_pcs,25,25
game_sessions,250,250
bar_orders,200,200


### Задание 2. Скалярные подзапросы в `WHERE` (Easy)
1. Сессии с оплатой выше среднего чека клуба.
2. Компьютеры с максимальной стоимостью часа.


In [3]:
# 2.1 Сессии с оплатой выше среднего чека клуба
q_e2_1 = '''
SELECT 
    session_id,
    game_title,
    hours_played,
    total_paid,
    ROUND((SELECT AVG(total_paid) FROM game_sessions), 2) AS avg_club_paid
FROM game_sessions
WHERE total_paid > (SELECT AVG(total_paid) FROM game_sessions)
ORDER BY total_paid DESC;
'''
df_res = pd.read_sql_query(q_e2_1, conn_easy)
print(f"Найдено сессий выше среднего: {len(df_res)}")
df_res.head()


Найдено сессий выше среднего: 104


session_id,game_title,hours_played,total_paid,avg_club_paid
10075,Valorant,5.0,2000,472.39
10133,GTA V,4.0,1600,472.39
10214,Counter-Strike 2,4.0,1600,472.39
10009,GTA V,4.0,1440,472.39
10049,GTA V,5.0,1350,472.39


In [4]:
# 2.2 Компьютеры с максимальной стоимостью часа
q_e2_2 = '''
SELECT pc_id, pc_number, zone_name, gpu, hourly_rate
FROM hardware_pcs
WHERE hourly_rate = (SELECT MAX(hourly_rate) FROM hardware_pcs);
'''
pd.read_sql_query(q_e2_2, conn_easy)


pc_id,pc_number,zone_name,gpu,hourly_rate
19,PC-19,VIP,RTX 4070 Ti,400
20,PC-20,VIP,RTX 4070 Ti,400
21,PC-21,VIP,RTX 4080,400
22,PC-22,VIP,RTX 4080,400


### Задание 3. Множественные подзапросы (`IN` и `NOT IN`)
1. Клиенты, игравшие в Counter-Strike 2 (`IN`).
2. Клиенты без заказов в баре (`NOT IN` с защитой от `NULL`).


In [5]:
# 3.1 Клиенты, игравшие в Counter-Strike 2
q_e3_1 = '''
SELECT member_id, nickname, city, loyalty_tier
FROM members
WHERE member_id IN (
    SELECT member_id 
    FROM game_sessions 
    WHERE game_title = 'Counter-Strike 2'
)
ORDER BY member_id;
'''
df_res = pd.read_sql_query(q_e3_1, conn_easy)
print(f"Игроков в CS2: {len(df_res)}")
df_res.head()


Игроков в CS2: 27


member_id,nickname,city,loyalty_tier
1,Phantom_715,Казань,Silver
4,Falcon_509,Санкт-Петербург,Gold
6,Frost_585,Санкт-Петербург,Bronze
7,Neo_949,Екатеринбург,Bronze
8,Phoenix_484,Санкт-Петербург,Bronze


In [6]:
# 3.2 Клиенты без заказов в баре (NOT IN с защитой от NULL)
q_e3_2 = '''
SELECT member_id, nickname, city, loyalty_tier
FROM members
WHERE member_id NOT IN (
    SELECT member_id 
    FROM bar_orders 
    WHERE member_id IS NOT NULL
)
ORDER BY member_id;
'''
df_res = pd.read_sql_query(q_e3_2, conn_easy)
print(f"Клиентов без покупок в баре: {len(df_res)}")
df_res.head(10)


Клиентов без покупок в баре: 45


member_id,nickname,city,loyalty_tier
1,Phantom_715,Казань,Silver
2,Sniper_880,Санкт-Петербург,Bronze
3,Dragon_387,Новосибирск,Bronze
4,Falcon_509,Санкт-Петербург,Gold
5,Titan_356,Москва,Bronze
6,Frost_585,Санкт-Петербург,Bronze
7,Neo_949,Екатеринбург,Bronze
8,Phoenix_484,Санкт-Петербург,Bronze
9,Wolf_710,Санкт-Петербург,Gold
10,Ronin_853,Санкт-Петербург,Bronze


### Задание 4. Коррелированные подзапросы в `WHERE` (Easy)
1. Сессии дольше средней длительности на данном компьютере.


In [7]:
# 4.1 Сессии дольше средней длительности на данном компьютере
q_e4_1 = '''
SELECT 
    s1.session_id,
    s1.pc_id,
    s1.game_title,
    s1.hours_played,
    s1.total_paid,
    ROUND((SELECT AVG(s2.hours_played) FROM game_sessions s2 WHERE s2.pc_id = s1.pc_id), 2) AS avg_hours_for_this_pc
FROM game_sessions s1
WHERE s1.hours_played > (
    SELECT AVG(s2.hours_played)
    FROM game_sessions s2
    WHERE s2.pc_id = s1.pc_id
)
ORDER BY s1.hours_played DESC;
'''
df_res = pd.read_sql_query(q_e4_1, conn_easy)
print(f"Сессий дольше среднего по ПК: {len(df_res)}")
df_res.head()


Сессий дольше среднего по ПК: 102


session_id,pc_id,game_title,hours_played,total_paid,avg_hours_for_this_pc
10049,17,GTA V,5.0,1350,2.40
10057,12,Valorant,5.0,650,1.92
10075,22,Valorant,5.0,2000,2.23
10098,5,GTA V,5.0,750,2.43
10140,4,GTA V,5.0,750,1.86


### Задание 5. Предикаты существования (`EXISTS` и `NOT EXISTS`)
1. Клиенты с покупками горячего питания (`EXISTS`).
2. Простаивающие ПК без единой сессии (`NOT EXISTS`).


In [8]:
# 5.1 Клиенты, бравшие горячее питание (EXISTS)
q_e5_1 = '''
SELECT m.member_id, m.nickname, m.city
FROM members m
WHERE EXISTS (
    SELECT 1 
    FROM bar_orders b 
    WHERE b.member_id = m.member_id AND b.category = 'Горячее'
)
ORDER BY m.member_id;
'''
df_res = pd.read_sql_query(q_e5_1, conn_easy)
print(f"Клиентов с покупками горячего: {len(df_res)}")
df_res.head()


Клиентов с покупками горячего: 30


member_id,nickname,city
40,Thunder_985,Новосибирск
41,Kraken_462,Новосибирск
43,Dragon_539,Екатеринбург
44,Rogue_172,Новосибирск
45,Neo_675,Новосибирск


In [9]:
# 5.2 Компьютеры без единой игровой сессии (NOT EXISTS)
q_e5_2 = '''
SELECT p.pc_id, p.pc_number, p.zone_name, p.hourly_rate
FROM hardware_pcs p
WHERE NOT EXISTS (
    SELECT 1 
    FROM game_sessions s 
    WHERE s.pc_id = p.pc_id
);
'''
df_res = pd.read_sql_query(q_e5_2, conn_easy)
print(f"Простаивающих ПК: {len(df_res)}")
df_res


Простаивающих ПК: 3


pc_id,pc_number,zone_name,hourly_rate
23,PC-23,PS5,300
24,PC-24,PS5,320
25,PC-25,PS5,300


### Задание 6. Подзапросы в секции `SELECT` (Easy)
1. Клиентская сводка трат (`sessions_count` и `bar_spend`).


In [10]:
# 6.1 Клиентская сводка трат
q_e6_1 = '''
SELECT 
    m.member_id,
    m.nickname,
    m.loyalty_tier,
    (SELECT COUNT(*) FROM game_sessions s WHERE s.member_id = m.member_id) AS sessions_count,
    (SELECT IFNULL(SUM(total_amount), 0) FROM bar_orders b WHERE b.member_id = m.member_id) AS bar_spend
FROM members m
ORDER BY bar_spend DESC;
'''
df_easy_summary = pd.read_sql_query(q_e6_1, conn_easy)
print("Топ-10 клиентов по тратам в баре:")
df_easy_summary.head(10)


Топ-10 клиентов по тратам в баре:


member_id,nickname,loyalty_tier,sessions_count,bar_spend
40,Thunder_985,Silver,0,2480
81,Slayer_522,Bronze,0,2440
41,Kraken_462,Silver,2,2010
93,Echo_386,Silver,0,1890
95,Kaiser_178,Bronze,0,1850
72,Hunter_451,Silver,1,1750
49,Rogue_214,Bronze,1,1740
76,Flash_968,Gold,0,1520
87,Dragon_939,Bronze,0,1490
68,Kaiser_612,Silver,3,1460


### Задание 7. Подзапросы в секции `FROM` (Агрегаты над агрегатами)
1. Среднее число сессий на одного игрока.
2. Средний суммарный барный чек гостя.


In [11]:
# 7.1 Среднее число сессий на игрока
q_e7_1 = '''
SELECT 
    ROUND(AVG(sessions_count), 2) AS avg_sessions_per_player,
    MIN(sessions_count) AS min_sessions,
    MAX(sessions_count) AS max_sessions
FROM (
    SELECT member_id, COUNT(*) AS sessions_count
    FROM game_sessions
    WHERE member_id IS NOT NULL
    GROUP BY member_id
) AS player_stat;
'''
pd.read_sql_query(q_e7_1, conn_easy)


avg_sessions_per_player,min_sessions,max_sessions
3.27,1,7


In [12]:
# 7.2 Средний барный LTV посетителя
q_e7_2 = '''
SELECT 
    ROUND(AVG(total_bar_spend), 2) AS avg_bar_ltv,
    MIN(total_bar_spend) AS min_bar_spend,
    MAX(total_bar_spend) AS max_bar_spend
FROM (
    SELECT member_id, SUM(total_amount) AS total_bar_spend
    FROM bar_orders
    WHERE member_id IS NOT NULL
    GROUP BY member_id
) AS bar_stat;
'''
pd.read_sql_query(q_e7_2, conn_easy)


avg_bar_ltv,min_bar_spend,max_bar_spend
873.18,110,2480


### Задание 8. Операции над множествами (`UNION`, `UNION ALL`, `INTERSECT`, `EXCEPT`)
1. Сравнение `UNION` vs `UNION ALL`.
2. `INTERSECT`: универсальные клиенты (ПК + бар).
3. `EXCEPT`: чистые геймеры (ПК без бара).


In [13]:
# 8.1 UNION vs UNION ALL
q_union = '''
SELECT member_id FROM game_sessions WHERE member_id IS NOT NULL
UNION
SELECT member_id FROM bar_orders WHERE member_id IS NOT NULL;
'''
q_union_all = '''
SELECT member_id FROM game_sessions WHERE member_id IS NOT NULL
UNION ALL
SELECT member_id FROM bar_orders WHERE member_id IS NOT NULL;
'''
df_u = pd.read_sql_query(q_union, conn_easy)
df_u_all = pd.read_sql_query(q_union_all, conn_easy)
print(f"Уникальных активных клиентов (UNION): {len(df_u)}")
print(f"Всего событий активности (UNION ALL): {len(df_u_all)}")


Уникальных активных клиентов (UNION): 91
Всего событий активности (UNION ALL): 407


In [14]:
# 8.2 INTERSECT: клиенты, которые и играли на ПК, и заказывали в баре
q_e8_2 = '''
SELECT m.member_id, m.nickname, m.city, m.loyalty_tier
FROM members m
WHERE m.member_id IN (
    SELECT member_id FROM game_sessions WHERE member_id IS NOT NULL
    INTERSECT
    SELECT member_id FROM bar_orders WHERE member_id IS NOT NULL
)
ORDER BY m.member_id;
'''
df_res = pd.read_sql_query(q_e8_2, conn_easy)
print(f"Универсальных гостей (ПК + Бар): {len(df_res)}")
df_res.head()


Универсальных гостей (ПК + Бар): 34


member_id,nickname,city,loyalty_tier
41,Kraken_462,Новосибирск,Silver
42,Cobra_604,Екатеринбург,Platinum
43,Dragon_539,Екатеринбург,Silver
44,Rogue_172,Новосибирск,Silver
45,Neo_675,Новосибирск,Gold


In [15]:
# 8.3 EXCEPT: чистые геймеры (играли на ПК, но ни разу не были в баре)
q_e8_3 = '''
SELECT m.member_id, m.nickname, m.city, m.loyalty_tier
FROM members m
WHERE m.member_id IN (
    SELECT member_id FROM game_sessions WHERE member_id IS NOT NULL
    EXCEPT
    SELECT member_id FROM bar_orders WHERE member_id IS NOT NULL
)
ORDER BY m.member_id;
'''
df_res = pd.read_sql_query(q_e8_3, conn_easy)
print(f"Чистых геймеров (без бара): {len(df_res)}")
df_res.head()


Чистых геймеров (без бара): 36


member_id,nickname,city,loyalty_tier
1,Phantom_715,Казань,Silver
2,Sniper_880,Санкт-Петербург,Bronze
3,Dragon_387,Новосибирск,Bronze
4,Falcon_509,Санкт-Петербург,Gold
5,Titan_356,Москва,Bronze


### Задание 9. Параметризованный отбор через плейсхолдеры `?`
1. Клиенты из выбранного города с депозитом выше порога.


In [16]:
# 9.1 Параметризованный отбор клиентов
city = "Москва"
min_bal = 500

q_e9_1 = '''
SELECT member_id, nickname, city, loyalty_tier, balance
FROM members
WHERE city = ? AND balance >= ?
ORDER BY balance DESC;
'''
df_res = pd.read_sql_query(q_e9_1, conn_easy, params=(city, min_bal))
print(f"Клиентов из {city} с балансом >= {min_bal} руб.: {len(df_res)}")
df_res.head()


Клиентов из Москва с балансом >= 500 руб.: 20


member_id,nickname,city,loyalty_tier,balance
14,Grizzly_890,Москва,Platinum,5000
100,Tempest_393,Москва,Bronze,5000
19,Apex_341,Москва,Bronze,3500
30,Shadow_236,Москва,Silver,3500
48,Shadow_801,Москва,Silver,3500


### Задание 10. Экспорт отчета в CSV (Easy)
Выгрузка сводки активности в `client_activity_report.csv`.


In [17]:
# Экспорт отчета
export_path_e = base_dir / "client_activity_report.csv"
df_easy_summary.to_csv(export_path_e, sep=";", index=False, encoding="utf-8")
print(f"[OK] Отчет активности Easy экспортирован: {export_path_e} ({len(df_easy_summary)} строк)")


[OK] Отчет активности Easy экспортирован: client_activity_report.csv (100 строк)


---
# 🟡 Часть 2. Эталонное решение Уровня Medium (Средний)
**5 таблиц:** `members`, `hardware_pcs`, `game_sessions`, `bar_orders`, `tournaments`

### Задание 1.1–1.6: Подготовка базы данных и миграция CSV (вариант 1)


In [18]:
# Загрузка 5 CSV-файлов Medium (вариант 1)
med_dir = base_dir / "medium" / "variant_01"
df_members_m = pd.read_csv(med_dir / "members.csv", sep=";")
df_pcs_m = pd.read_csv(med_dir / "hardware_pcs.csv", sep=";")
df_sessions_m = pd.read_csv(med_dir / "game_sessions.csv", sep=";")
df_bar_m = pd.read_csv(med_dir / "bar_orders.csv", sep=";")
df_tourn_m = pd.read_csv(med_dir / "tournaments.csv", sep=";")

conn_med = sqlite3.connect(":memory:")
conn_med.execute("PRAGMA foreign_keys = ON;")
cur_m = conn_med.cursor()

cur_m.executescript('''
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    nickname TEXT NOT NULL,
    city TEXT NOT NULL,
    email TEXT,
    registered_at TEXT NOT NULL,
    loyalty_tier TEXT NOT NULL,
    balance INTEGER NOT NULL
);

CREATE TABLE hardware_pcs (
    pc_id INTEGER PRIMARY KEY,
    pc_number TEXT NOT NULL,
    zone_name TEXT NOT NULL,
    hourly_rate INTEGER NOT NULL,
    gpu TEXT NOT NULL,
    cpu TEXT NOT NULL,
    ram_gb INTEGER NOT NULL,
    status TEXT NOT NULL
);

CREATE TABLE game_sessions (
    session_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES members(member_id),
    pc_id INTEGER REFERENCES hardware_pcs(pc_id),
    game_title TEXT NOT NULL,
    session_date TEXT NOT NULL,
    hours_played REAL NOT NULL,
    discount_promo INTEGER NOT NULL,
    total_paid INTEGER NOT NULL,
    client_rating INTEGER
);

CREATE TABLE bar_orders (
    order_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES members(member_id),
    item_name TEXT NOT NULL,
    category TEXT NOT NULL,
    order_date TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    item_price INTEGER NOT NULL,
    total_amount INTEGER NOT NULL
);

CREATE TABLE tournaments (
    tournament_id INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title TEXT NOT NULL,
    prize_fund INTEGER NOT NULL,
    entry_fee INTEGER NOT NULL,
    start_date TEXT NOT NULL,
    min_rank TEXT NOT NULL
);
''')

# Импорт данных
df_members_m.to_sql("members", conn_med, if_exists="append", index=False)
df_pcs_m.to_sql("hardware_pcs", conn_med, if_exists="append", index=False)
df_sessions_m.to_sql("game_sessions", conn_med, if_exists="append", index=False)
df_bar_m.to_sql("bar_orders", conn_med, if_exists="append", index=False)
df_tourn_m.to_sql("tournaments", conn_med, if_exists="append", index=False)
conn_med.commit()

audit_m_query = '''
SELECT 'members' AS tbl, COUNT(*) AS sql_cnt, ? AS pd_cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*), ? FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*), ? FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*), ? FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*), ? FROM tournaments;
'''
pd.read_sql_query(audit_m_query, conn_med, params=(len(df_members_m), len(df_pcs_m), len(df_sessions_m), len(df_bar_m), len(df_tourn_m)))


tbl,sql_cnt,pd_cnt
members,150,150
hardware_pcs,30,30
game_sessions,500,500
bar_orders,350,350
tournaments,7,7


### Задание 2. Скалярные подзапросы в `WHERE` (Medium)
1. Сессии по конкретной игре (`Dota 2`), сумма оплаты которых выше среднего чека по этой игре.


In [19]:
# 2.1 Сессии дороже среднего чека по игре Dota 2
q_m2_1 = '''
SELECT 
    session_id,
    game_title,
    hours_played,
    total_paid,
    ROUND((SELECT AVG(total_paid) FROM game_sessions WHERE game_title = 'Dota 2'), 2) AS avg_dota_paid
FROM game_sessions
WHERE game_title = 'Dota 2'
  AND total_paid > (SELECT AVG(total_paid) FROM game_sessions WHERE game_title = 'Dota 2')
ORDER BY total_paid DESC;
'''
df_res = pd.read_sql_query(q_m2_1, conn_med)
print(f"Сессий Dota 2 выше среднего по игре: {len(df_res)}")
df_res.head()


Сессий Dota 2 выше среднего по игре: 26


session_id,game_title,hours_played,total_paid,avg_dota_paid
20445,Dota 2,3.0,1200,458.74
20141,Dota 2,4.0,1000,458.74
20435,Dota 2,4.0,1000,458.74
20411,Dota 2,6.0,900,458.74
20390,Dota 2,3.0,810,458.74


### Задание 3. Множественные подзапросы (`IN` и `NOT IN`)
1. Клиенты, игравшие в турнирные дисциплины (`IN (SELECT game_title FROM tournaments)`).
2. Клиенты без заказов в категории `'Напитки'` (`NOT IN` с защитой от `NULL`).


In [20]:
# 3.1 Клиенты, игравшие в официальные турнирные дисциплины
q_m3_1 = '''
SELECT DISTINCT m.member_id, m.nickname, m.city, m.loyalty_tier
FROM members m
WHERE m.member_id IN (
    SELECT s.member_id 
    FROM game_sessions s 
    WHERE s.game_title IN (SELECT game_title FROM tournaments)
)
ORDER BY m.member_id;
'''
df_res = pd.read_sql_query(q_m3_1, conn_med)
print(f"Игроков турнирных дисциплин: {len(df_res)}")
df_res.head()


Игроков турнирных дисциплин: 105


member_id,nickname,city,loyalty_tier
1,Zero_659,Санкт-Петербург,Silver
2,Vortex_745,Москва,Bronze
3,Bullet_419,Москва,Bronze
4,Phantom_544,Новосибирск,Bronze
5,Viper_480,Казань,Platinum


In [21]:
# 3.2 Клиенты без единого заказа напитков в баре (NOT IN)
q_m3_2 = '''
SELECT member_id, nickname, city, loyalty_tier
FROM members
WHERE member_id NOT IN (
    SELECT member_id 
    FROM bar_orders 
    WHERE category = 'Напитки' AND member_id IS NOT NULL
)
ORDER BY member_id;
'''
df_res = pd.read_sql_query(q_m3_2, conn_med)
print(f"Клиентов без покупок напитков: {len(df_res)}")
df_res.head()


Клиентов без покупок напитков: 86


member_id,nickname,city,loyalty_tier
1,Zero_659,Санкт-Петербург,Silver
2,Vortex_745,Москва,Bronze
3,Bullet_419,Москва,Bronze
4,Phantom_544,Новосибирск,Bronze
5,Viper_480,Казань,Platinum


### Задание 4. Коррелированные подзапросы в `WHERE` (Medium)
1. Аномально длинные сессии по конкретным ПК.
2. Крупные барные чеки внутри своей категории товаров.


In [22]:
# 4.1 Аномально длинные сессии по ПК
q_m4_1 = '''
SELECT 
    s1.session_id,
    s1.pc_id,
    s1.game_title,
    s1.hours_played,
    s1.total_paid,
    ROUND((SELECT AVG(s2.hours_played) FROM game_sessions s2 WHERE s2.pc_id = s1.pc_id), 2) AS avg_hours_for_pc
FROM game_sessions s1
WHERE s1.hours_played > (
    SELECT AVG(s2.hours_played)
    FROM game_sessions s2
    WHERE s2.pc_id = s1.pc_id
)
ORDER BY s1.hours_played DESC;
'''
df_res = pd.read_sql_query(q_m4_1, conn_med)
print(f"Аномально длинных сессий: {len(df_res)}")
df_res.head()


Аномально длинных сессий: 206


session_id,pc_id,game_title,hours_played,total_paid,avg_hours_for_pc
20029,3,Counter-Strike 2,6.0,780,2.15
20132,17,Cyberpunk 2077,6.0,1500,2.44
20274,24,Cyberpunk 2077,6.0,2280,2.18
20308,26,Cyberpunk 2077,6.0,2400,2.37
20411,6,Dota 2,6.0,900,2.21


In [23]:
# 4.2 Крупные заказы внутри своей категории
q_m4_2 = '''
SELECT 
    b1.order_id,
    b1.category,
    b1.item_name,
    b1.total_amount,
    ROUND((SELECT AVG(b2.total_amount) FROM bar_orders b2 WHERE b2.category = b1.category), 2) AS avg_cat_amount
FROM bar_orders b1
WHERE b1.total_amount > (
    SELECT AVG(b2.total_amount)
    FROM bar_orders b2
    WHERE b2.category = b1.category
)
ORDER BY b1.category, b1.total_amount DESC;
'''
df_res = pd.read_sql_query(q_m4_2, conn_med)
print(f"Крупных заказов в категориях: {len(df_res)}")
df_res.head()


Крупных заказов в категориях: 121


order_id,category,item_name,total_amount,avg_cat_amount
6042,Горячее,Бургер Классический,1020,432.11
6169,Горячее,Бургер Классический,1020,432.11
6063,Горячее,Сэндвич с ветчиной и сыром,780,432.11
6056,Горячее,Наггетсы куриные (8 шт),750,432.11
6133,Горячее,Наггетсы куриные (8 шт),750,432.11


### Задание 5. Предикаты существования (`EXISTS` и `NOT EXISTS`)
1. Спящие клиенты без единой игровой сессии (`NOT EXISTS`).
2. Турниры, подкрепленные реальными сессиями гостей (`EXISTS`).


In [24]:
# 5.1 Спящие клиенты (NOT EXISTS)
q_m5_1 = '''
SELECT m.member_id, m.nickname, m.city, m.loyalty_tier
FROM members m
WHERE NOT EXISTS (
    SELECT 1 
    FROM game_sessions s 
    WHERE s.member_id = m.member_id
)
ORDER BY m.member_id;
'''
df_res = pd.read_sql_query(q_m5_1, conn_med)
print(f"Спящих клиентов: {len(df_res)}")
df_res.head()


Спящих клиентов: 31


member_id,nickname,city,loyalty_tier
75,Titan_145,Москва,Gold
121,Falcon_720,Новосибирск,Bronze
122,Phantom_47,Москва,Bronze
123,Neo_807,Новосибирск,Bronze
124,Turbo_729,Санкт-Петербург,Gold


In [25]:
# 5.2 Турниры, по дисциплинам которых играли гости (EXISTS)
q_m5_2 = '''
SELECT t.tournament_id, t.tournament_name, t.game_title, t.prize_fund
FROM tournaments t
WHERE EXISTS (
    SELECT 1 
    FROM game_sessions s 
    WHERE s.game_title = t.game_title
);
'''
pd.read_sql_query(q_m5_2, conn_med)


tournament_id,tournament_name,game_title,prize_fund
101,Winter Cyber Cup: CS2,Counter-Strike 2,50000
102,Spring DOTA 2 Major,Dota 2,75000
103,Valorant City Clash,Valorant,35000
104,Apex Legends Trio Battle,Apex Legends,40000
105,Cyberpunk Night Runner Solo,Cyberpunk 2077,25000
106,Pro Masters Elite: CS2,Counter-Strike 2,120000
107,Dota 2 Weekend Brawl,Dota 2,30000


### Задание 6. Подзапросы в секции `SELECT` (Аналитика «Клиент 360»)
Комплексный профиль клиента без риска декартова произведения.


In [26]:
# 6.1 Аналитическая витрина «Клиент 360»
q_client_360 = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    m.loyalty_tier,
    (SELECT COUNT(*) FROM game_sessions s WHERE s.member_id = m.member_id) AS sessions_count,
    (SELECT IFNULL(SUM(total_paid), 0) FROM game_sessions s WHERE s.member_id = m.member_id) AS game_spend,
    (SELECT IFNULL(SUM(total_amount), 0) FROM bar_orders b WHERE b.member_id = m.member_id) AS bar_spend,
    ((SELECT IFNULL(SUM(total_paid), 0) FROM game_sessions s WHERE s.member_id = m.member_id) +
     (SELECT IFNULL(SUM(total_amount), 0) FROM bar_orders b WHERE b.member_id = m.member_id)) AS total_revenue
FROM members m
ORDER BY game_spend DESC;
'''
df_client_360 = pd.read_sql_query(q_client_360, conn_med)
print("Топ-15 клиентов по игровым тратам:")
df_client_360.head(15)


Топ-15 клиентов по игровым тратам:


member_id,nickname,city,loyalty_tier,sessions_count,game_spend,bar_spend,total_revenue
85,Wolf_460,Москва,Silver,12,6500,1480,7980
95,Turbo_24,Екатеринбург,Bronze,8,5722,2935,8657
116,Samurai_621,Казань,Bronze,8,5703,675,6378
6,Frost_799,Санкт-Петербург,Bronze,9,4949,0,4949
115,Thunder_100,Новосибирск,Platinum,9,4907,1220,6127
56,Frost_603,Новосибирск,Platinum,7,4225,1510,5735
88,Storm_326,Москва,Silver,8,4078,160,4238
4,Phantom_544,Новосибирск,Bronze,8,3884,0,3884
104,Bullet_297,Москва,Silver,6,3857,0,3857
26,Lynx_453,Санкт-Петербург,Platinum,7,3685,0,3685


### Задание 7. Подзапросы в секции `FROM` (Medium)
Высокодоходные категории бара со средним чеком выше общего среднего чека бара.


In [27]:
# 7.1 Категории бара выше среднего чека бара
q_m7_1 = '''
SELECT 
    category,
    avg_cat_check,
    (SELECT ROUND(AVG(total_amount), 2) FROM bar_orders) AS club_avg_bar_check
FROM (
    SELECT category, ROUND(AVG(total_amount), 2) AS avg_cat_check
    FROM bar_orders
    GROUP BY category
) AS cat_stats
WHERE avg_cat_check > (SELECT AVG(total_amount) FROM bar_orders)
ORDER BY avg_cat_check DESC;
'''
pd.read_sql_query(q_m7_1, conn_med)


category,avg_cat_check,club_avg_bar_check
Мерч,1674.65,559.43


### Задание 8. Кросс-табличная аналитика и операции над множествами (5 таблиц)
1. `INTERSECT`: турнирные дисциплины клуба.
2. `EXCEPT`: игры на ПК вне киберспорта.
3. `UNION ALL`: сквозная финансовая лента из 3 источников.


In [28]:
# 8.1 INTERSECT: официальные дисциплины турниров, в которые играют гости
q_m8_1 = '''
SELECT DISTINCT game_title FROM game_sessions
INTERSECT
SELECT DISTINCT game_title FROM tournaments;
'''
pd.read_sql_query(q_m8_1, conn_med)


game_title
Apex Legends
Counter-Strike 2
Cyberpunk 2077
Dota 2
Valorant


In [29]:
# 8.2 EXCEPT: игры клуба, по которым турниры НЕ проводятся
q_m8_2 = '''
SELECT DISTINCT game_title FROM game_sessions
EXCEPT
SELECT DISTINCT game_title FROM tournaments;
'''
pd.read_sql_query(q_m8_2, conn_med)


game_title
Atomic Heart
GTA V
PUBG
Rust
Warzone


In [30]:
# 8.3 UNION ALL: сквозная финансовая лента клуба
q_m8_3 = '''
SELECT 'Аренда ПК' AS source_type, session_date AS event_date, total_paid AS amount
FROM game_sessions
UNION ALL
SELECT 'Бар клуба', order_date, total_amount
FROM bar_orders
UNION ALL
SELECT 'Турнир (Фонд)', start_date, prize_fund
FROM tournaments
ORDER BY event_date DESC
LIMIT 15;
'''
pd.read_sql_query(q_m8_3, conn_med)


source_type,event_date,amount
Турнир (Фонд),2026-05-15,30000
Турнир (Фонд),2026-05-01,120000
Турнир (Фонд),2026-04-18,25000
Турнир (Фонд),2026-04-05,40000
Бар клуба,2026-04-01 00:15:00,220
Бар клуба,2026-03-31 22:30:00,480
Бар клуба,2026-03-31 21:45:00,180
Аренда ПК,2026-03-31 19:15:00,1000
Бар клуба,2026-03-31 16:00:00,140
Аренда ПК,2026-03-31 13:45:00,540


### Задание 9. Параметризованный отбор хайроллеров (WHERE + ?)
Отбор клиентов из городов `(?, ?)` с тратами выше порогового значения `?`.


In [31]:
# 9.1 Параметризованный отбор хайроллеров
threshold = 3000
cities = ('Москва', 'Санкт-Петербург')

q_m9_1 = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    m.loyalty_tier,
    ((SELECT IFNULL(SUM(total_paid), 0) FROM game_sessions s WHERE s.member_id = m.member_id) +
     (SELECT IFNULL(SUM(total_amount), 0) FROM bar_orders b WHERE b.member_id = m.member_id)) AS total_spend
FROM members m
WHERE m.city IN (?, ?)
  AND ((SELECT IFNULL(SUM(total_paid), 0) FROM game_sessions s WHERE s.member_id = m.member_id) +
       (SELECT IFNULL(SUM(total_amount), 0) FROM bar_orders b WHERE b.member_id = m.member_id)) > ?
ORDER BY total_spend DESC;
'''
df_highrollers = pd.read_sql_query(q_m9_1, conn_med, params=(cities[0], cities[1], threshold))
print(f"Хайроллеров из Москвы и СПб (траты > {threshold} руб.): {len(df_highrollers)}")
df_highrollers.head()


Хайроллеров из Москвы и СПб (траты > 3000 руб.): 30


member_id,nickname,city,loyalty_tier,total_spend
85,Wolf_460,Москва,Silver,7980
89,Raven_747,Санкт-Петербург,Gold,7742
55,Tempest_897,Москва,Gold,7584
50,Rogue_812,Санкт-Петербург,Bronze,7230
73,Zero_355,Санкт-Петербург,Gold,7063


### Задание 10. Экспорт аналитических срезов (Medium)
Выгрузка витрины «Клиент 360» в `client_360_analytics.csv`.


In [32]:
# Экспорт витрины Medium
export_path_m = base_dir / "client_360_analytics.csv"
df_client_360.to_csv(export_path_m, sep=";", index=False, encoding="utf-8")
print(f"[OK] Аналитическая витрина Medium экспортирована: {export_path_m} ({len(df_client_360)} строк)")


[OK] Аналитическая витрина Medium экспортирована: client_360_analytics.csv (150 строк)


---
# 🔴 Часть 3. Эталонное решение Уровня Hard (Продвинутый)
**6 таблиц:** `members`, `hardware_pcs`, `game_sessions`, `bar_orders`, `tournaments`, `tournament_entries`

### Задание 1.1–1.6: Архитектура схемы и валидация 6 таблиц (вариант 1)


In [33]:
# Загрузка 6 CSV-файлов Hard (вариант 1)
hard_dir = base_dir / "hard" / "variant_01"
df_members_h = pd.read_csv(hard_dir / "members.csv", sep=";")
df_pcs_h = pd.read_csv(hard_dir / "hardware_pcs.csv", sep=";")
df_sessions_h = pd.read_csv(hard_dir / "game_sessions.csv", sep=";")
df_bar_h = pd.read_csv(hard_dir / "bar_orders.csv", sep=";")
df_tourn_h = pd.read_csv(hard_dir / "tournaments.csv", sep=";")
df_entries_h = pd.read_csv(hard_dir / "tournament_entries.csv", sep=";")

conn_hard = sqlite3.connect(":memory:")
conn_hard.execute("PRAGMA foreign_keys = ON;")
cur_h = conn_hard.cursor()

cur_h.executescript('''
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    nickname TEXT NOT NULL,
    city TEXT NOT NULL,
    email TEXT,
    registered_at TEXT NOT NULL,
    loyalty_tier TEXT NOT NULL,
    balance INTEGER NOT NULL
);

CREATE TABLE hardware_pcs (
    pc_id INTEGER PRIMARY KEY,
    pc_number TEXT NOT NULL,
    zone_name TEXT NOT NULL,
    hourly_rate INTEGER NOT NULL,
    gpu TEXT NOT NULL,
    cpu TEXT NOT NULL,
    ram_gb INTEGER NOT NULL,
    status TEXT NOT NULL
);

CREATE TABLE game_sessions (
    session_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES members(member_id),
    pc_id INTEGER REFERENCES hardware_pcs(pc_id),
    game_title TEXT NOT NULL,
    session_date TEXT NOT NULL,
    hours_played REAL NOT NULL,
    discount_promo INTEGER NOT NULL,
    total_paid INTEGER NOT NULL,
    client_rating INTEGER
);

CREATE TABLE bar_orders (
    order_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES members(member_id),
    item_name TEXT NOT NULL,
    category TEXT NOT NULL,
    order_date TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    item_price INTEGER NOT NULL,
    total_amount INTEGER NOT NULL
);

CREATE TABLE tournaments (
    tournament_id INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title TEXT NOT NULL,
    prize_fund INTEGER NOT NULL,
    entry_fee INTEGER NOT NULL,
    start_date TEXT NOT NULL,
    min_rank TEXT NOT NULL
);

CREATE TABLE tournament_entries (
    entry_id INTEGER PRIMARY KEY,
    tournament_id INTEGER REFERENCES tournaments(tournament_id),
    member_id INTEGER REFERENCES members(member_id),
    team_name TEXT NOT NULL,
    placement INTEGER NOT NULL,
    prize_won INTEGER NOT NULL
);
''')

# Импорт данных
df_members_h.to_sql("members", conn_hard, if_exists="append", index=False)
df_pcs_h.to_sql("hardware_pcs", conn_hard, if_exists="append", index=False)
df_sessions_h.to_sql("game_sessions", conn_hard, if_exists="append", index=False)
df_bar_h.to_sql("bar_orders", conn_hard, if_exists="append", index=False)
df_tourn_h.to_sql("tournaments", conn_hard, if_exists="append", index=False)
df_entries_h.to_sql("tournament_entries", conn_hard, if_exists="append", index=False)
conn_hard.commit()

audit_h_query = '''
SELECT 'members' AS tbl, COUNT(*) AS sql_cnt, ? AS pd_cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*), ? FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*), ? FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*), ? FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*), ? FROM tournaments
UNION ALL SELECT 'tournament_entries', COUNT(*), ? FROM tournament_entries;
'''
pd.read_sql_query(audit_h_query, conn_hard, params=(
    len(df_members_h), len(df_pcs_h), len(df_sessions_h), 
    len(df_bar_h), len(df_tourn_h), len(df_entries_h)
))


tbl,sql_cnt,pd_cnt
members,200,200
hardware_pcs,40,40
game_sessions,800,800
bar_orders,500,500
tournaments,7,7
tournament_entries,120,120


### Задание 2. Скалярные подзапросы в `WHERE` (Hard)
1. Рекордные призовые: выигрыш выше среднего призового фонда всех турниров клуба.


In [34]:
# 2.1 Рекордные призовые выплаты
q_h2_1 = '''
SELECT 
    te.entry_id,
    te.tournament_id,
    te.member_id,
    te.team_name,
    te.placement,
    te.prize_won,
    (SELECT ROUND(AVG(prize_fund), 2) FROM tournaments) AS avg_tournament_fund
FROM tournament_entries te
WHERE te.prize_won > (SELECT AVG(prize_fund) FROM tournaments)
ORDER BY te.prize_won DESC;
'''
df_res = pd.read_sql_query(q_h2_1, conn_hard)
print(f"Рекордных призовых выплат: {len(df_res)}")
df_res


Рекордных призовых выплат: 1


entry_id,tournament_id,member_id,team_name,placement,prize_won,avg_tournament_fund
8084,106,67,Cyber Team,1,60000,53571.43


### Задание 3. Множественные подзапросы (`IN` и `NOT IN`)
1. Участники турниров, которые ни разу не заняли призовое место $\le 3$.


In [35]:
# 3.1 Участники турниров без призовых мест (топ-3)
q_h3_1 = '''
SELECT member_id, nickname, city, loyalty_tier
FROM members
WHERE member_id IN (SELECT member_id FROM tournament_entries)
  AND member_id NOT IN (
      SELECT member_id 
      FROM tournament_entries 
      WHERE placement <= 3 AND member_id IS NOT NULL
  )
ORDER BY member_id;
'''
df_res = pd.read_sql_query(q_h3_1, conn_hard)
print(f"Участников турниров без призовых мест: {len(df_res)}")
df_res.head()


Участников турниров без призовых мест: 59


member_id,nickname,city,loyalty_tier
2,Samurai_704,Москва,Gold
4,Wolf_945,Санкт-Петербург,Platinum
5,Valkyrie_443,Санкт-Петербург,Silver
8,Frost_730,Новосибирск,Bronze
9,Neo_792,Санкт-Петербург,Silver


### Задание 4. Глубокие коррелированные подзапросы (Hard)
1. Сессии дольше средней длительности внутри посадочной зоны ПК.
2. Выдающиеся выигрыши внутри соответствующего турнира.


In [36]:
# 4.1 Сессии дольше средней длительности внутри своей посадочной зоны
q_h4_1 = '''
SELECT 
    s1.session_id,
    p1.zone_name,
    s1.game_title,
    s1.hours_played,
    s1.total_paid,
    ROUND((
        SELECT AVG(s2.hours_played)
        FROM game_sessions s2
        JOIN hardware_pcs p2 ON s2.pc_id = p2.pc_id
        WHERE p2.zone_name = p1.zone_name
    ), 2) AS avg_zone_hours
FROM game_sessions s1
JOIN hardware_pcs p1 ON s1.pc_id = p1.pc_id
WHERE s1.hours_played > (
    SELECT AVG(s2.hours_played)
    FROM game_sessions s2
    JOIN hardware_pcs p2 ON s2.pc_id = p2.pc_id
    WHERE p2.zone_name = p1.zone_name
)
ORDER BY s1.hours_played DESC;
'''
df_res = pd.read_sql_query(q_h4_1, conn_hard)
print(f"Премиальных сессий по зонам: {len(df_res)}")
df_res.head()


Премиальных сессий по зонам: 328


session_id,zone_name,game_title,hours_played,total_paid,avg_zone_hours
30020,VIP,PUBG,6.0,2376,2.21
30072,Bootcamp,Atomic Heart,6.0,1440,2.23
30073,Standard,Rust,6.0,840,2.33
30076,Standard,Rust,6.0,840,2.33
30171,Standard,Cyberpunk 2077,6.0,1080,2.33


In [37]:
# 4.2 Выдающиеся выигрыши внутри соответствующего турнира
q_h4_2 = '''
SELECT 
    te1.entry_id,
    te1.tournament_id,
    te1.member_id,
    te1.placement,
    te1.prize_won,
    ROUND((
        SELECT AVG(te2.prize_won)
        FROM tournament_entries te2
        WHERE te2.tournament_id = te1.tournament_id AND te2.prize_won > 0
    ), 2) AS avg_tournament_prize
FROM tournament_entries te1
WHERE te1.prize_won > (
    SELECT AVG(te2.prize_won)
    FROM tournament_entries te2
    WHERE te2.tournament_id = te1.tournament_id AND te2.prize_won > 0
)
ORDER BY te1.prize_won DESC;
'''
df_res = pd.read_sql_query(q_h4_2, conn_hard)
print(f"Выигрышей выше среднего по турниру: {len(df_res)}")
df_res.head()


Выигрышей выше среднего по турниру: 7


entry_id,tournament_id,member_id,placement,prize_won,avg_tournament_prize
8084,106,67,1,60000,40000.00
8018,102,20,1,37500,25000.00
8001,101,99,1,25000,16666.67
8050,104,29,1,20000,13333.33
8032,103,30,1,17500,11666.67


### Задание 5. Сложная логика предикатов существования (`EXISTS` и `NOT EXISTS`)
1. Клиенты-«триатлонисты» (тройной `EXISTS`: ПК + бар + турниры).
2. Скрытые чемпионы: призеры турниров без единого заказа в баре.


In [38]:
# 5.1 Клиенты-«триатлонисты» (тройной EXISTS)
q_h5_1 = '''
SELECT m.member_id, m.nickname, m.city, m.loyalty_tier
FROM members m
WHERE EXISTS (SELECT 1 FROM game_sessions s WHERE s.member_id = m.member_id)
  AND EXISTS (SELECT 1 FROM bar_orders b WHERE b.member_id = m.member_id)
  AND EXISTS (SELECT 1 FROM tournament_entries te WHERE te.member_id = m.member_id)
ORDER BY m.member_id;
'''
df_triathlon = pd.read_sql_query(q_h5_1, conn_hard)
print(f"Универсальных клиентов-триатлонистов: {len(df_triathlon)}")
df_triathlon.head(10)


Универсальных клиентов-триатлонистов: 37


member_id,nickname,city,loyalty_tier
70,Samurai_735,Санкт-Петербург,Gold
71,Specter_590,Москва,Gold
73,Frost_65,Санкт-Петербург,Bronze
78,Slayer_761,Новосибирск,Gold
79,Neo_716,Санкт-Петербург,Gold
80,Specter_811,Казань,Bronze
82,Sniper_384,Санкт-Петербург,Gold
84,Dragon_864,Москва,Silver
85,Thunder_822,Москва,Gold
86,Phoenix_364,Москва,Gold


In [39]:
# 5.2 Скрытые чемпионы: призеры турнира (место 1-3), которые ни разу не брали ничего в баре
q_h5_2 = '''
SELECT DISTINCT m.member_id, m.nickname, m.city, m.loyalty_tier
FROM members m
WHERE EXISTS (
    SELECT 1 
    FROM tournament_entries te 
    WHERE te.member_id = m.member_id AND te.placement <= 3
)
AND NOT EXISTS (
    SELECT 1 
    FROM bar_orders b 
    WHERE b.member_id = m.member_id
)
ORDER BY m.member_id;
'''
df_hidden_champs = pd.read_sql_query(q_h5_2, conn_hard)
print(f"Скрытых чемпионов (призеры без бара): {len(df_hidden_champs)}")
df_hidden_champs


Скрытых чемпионов (призеры без бара): 13


member_id,nickname,city,loyalty_tier
6,Turbo_959,Санкт-Петербург,Gold
20,Apex_651,Санкт-Петербург,Silver
23,Rogue_201,Москва,Silver
24,Kaiser_477,Санкт-Петербург,Bronze
29,Thunder_342,Москва,Bronze
30,Tempest_458,Москва,Platinum
36,Phantom_385,Москва,Gold
40,Ronin_617,Санкт-Петербург,Gold
42,Wolf_18,Москва,Bronze
53,Sniper_457,Казань,Bronze


### Задание 6. Подзапросы в секции `SELECT` (Полный расчет Net LTV)
Расчет чистой маржи клуба с каждого клиента по 6 таблицам.


In [40]:
# 6.1 Расчет Net LTV и выявление топ-15 наиболее прибыльных клиентов
q_net_ltv = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    m.loyalty_tier,
    (SELECT IFNULL(SUM(total_paid), 0) FROM game_sessions s WHERE s.member_id = m.member_id) AS game_spend,
    (SELECT IFNULL(SUM(total_amount), 0) FROM bar_orders b WHERE b.member_id = m.member_id) AS bar_spend,
    (SELECT IFNULL(SUM(t.entry_fee), 0) 
     FROM tournament_entries te 
     JOIN tournaments t ON te.tournament_id = t.tournament_id 
     WHERE te.member_id = m.member_id) AS tournaments_fee,
    (SELECT IFNULL(SUM(prize_won), 0) FROM tournament_entries te WHERE te.member_id = m.member_id) AS prizes_earned,
    ((SELECT IFNULL(SUM(total_paid), 0) FROM game_sessions s WHERE s.member_id = m.member_id) +
     (SELECT IFNULL(SUM(total_amount), 0) FROM bar_orders b WHERE b.member_id = m.member_id) +
     (SELECT IFNULL(SUM(t.entry_fee), 0) 
      FROM tournament_entries te 
      JOIN tournaments t ON te.tournament_id = t.tournament_id 
      WHERE te.member_id = m.member_id) -
     (SELECT IFNULL(SUM(prize_won), 0) FROM tournament_entries te WHERE te.member_id = m.member_id)
    ) AS net_profit
FROM members m
ORDER BY net_profit DESC;
'''
df_net_ltv = pd.read_sql_query(q_net_ltv, conn_hard)
print("Топ-15 клиентов по чистой доходности (Net Profit):")
df_net_ltv.head(15)


Топ-15 клиентов по чистой доходности (Net Profit):


member_id,nickname,city,loyalty_tier,game_spend,bar_spend,tournaments_fee,prizes_earned,net_profit
72,Slayer_881,Москва,Gold,2953,8580,0,0,11533
145,Dragon_596,Санкт-Петербург,Silver,4624,6350,0,0,10974
146,CyberKnight_667,Москва,Silver,7370,2960,0,0,10330
96,Neo_347,Москва,Bronze,5412,4360,300,0,10072
73,Frost_65,Санкт-Петербург,Bronze,3276,5945,400,0,9621
108,Titan_769,Москва,Bronze,3300,5980,0,0,9280
150,Shadow_10,Москва,Silver,3894,5070,0,0,8964
135,Falcon_483,Санкт-Петербург,Gold,7493,1470,0,0,8963
103,Frost_890,Москва,Bronze,4250,2550,2050,0,8850
144,Reaper_993,Москва,Bronze,1990,6805,0,0,8795


### Задание 7. Подзапросы в секции `FROM` (Многомерные производные таблицы)
1. Дисперсия активности игроков.
2. Зоны повышенной доходности клуба.


In [41]:
# 7.1 Дисперсия активности игроков
q_h7_1 = '''
SELECT 
    ROUND(AVG(sessions_per_user), 2) AS avg_sessions,
    MIN(sessions_per_user) AS min_sessions,
    MAX(sessions_per_user) AS max_sessions,
    ROUND(AVG(total_hours), 2) AS avg_hours,
    ROUND(AVG(total_game_paid), 2) AS avg_game_ltv
FROM (
    SELECT 
        member_id, 
        COUNT(*) AS sessions_per_user,
        SUM(hours_played) AS total_hours,
        SUM(total_paid) AS total_game_paid
    FROM game_sessions
    WHERE member_id IS NOT NULL
    GROUP BY member_id
) AS active_players;
'''
pd.read_sql_query(q_h7_1, conn_hard)


avg_sessions,min_sessions,max_sessions,avg_hours,avg_game_ltv
4.53,1,13,10.35,2404.63


In [42]:
# 7.2 Зоны повышенной доходности клуба
q_h7_2 = '''
SELECT 
    zone_name,
    avg_zone_check,
    (SELECT ROUND(AVG(total_paid), 2) FROM game_sessions) AS club_avg_check
FROM (
    SELECT 
        p.zone_name,
        ROUND(AVG(s.total_paid), 2) AS avg_zone_check
    FROM game_sessions s
    JOIN hardware_pcs p ON s.pc_id = p.pc_id
    GROUP BY p.zone_name
) AS zone_stats
WHERE avg_zone_check > (SELECT AVG(total_paid) FROM game_sessions)
ORDER BY avg_zone_check DESC;
'''
pd.read_sql_query(q_h7_2, conn_hard)


zone_name,avg_zone_check,club_avg_check
VIP,916.92,526.52
Bootcamp,576.38,526.52


### Задание 8. Операции над множествами на 6 таблицах (`UNION ALL`, `EXCEPT`)
1. `EXCEPT`: участники турниров без барных трат.
2. `UNION ALL`: детальный 4-поточный финансовый аудит с расчетом сальдо.


In [43]:
# 8.1 EXCEPT: участники турниров, ни разу не бравшие ничего в баре
q_h8_1 = '''
SELECT m.member_id, m.nickname, m.city, m.loyalty_tier
FROM members m
WHERE m.member_id IN (
    SELECT member_id FROM tournament_entries WHERE member_id IS NOT NULL
    EXCEPT
    SELECT member_id FROM bar_orders WHERE member_id IS NOT NULL
)
ORDER BY m.member_id;
'''
df_res = pd.read_sql_query(q_h8_1, conn_hard)
print(f"Турнирных игроков без бара: {len(df_res)}")
df_res.head()


Турнирных игроков без бара: 40


member_id,nickname,city,loyalty_tier
2,Samurai_704,Москва,Gold
4,Wolf_945,Санкт-Петербург,Platinum
5,Valkyrie_443,Санкт-Петербург,Silver
6,Turbo_959,Санкт-Петербург,Gold
8,Frost_730,Новосибирск,Bronze


In [44]:
# 8.2 UNION ALL: детальный аудит 4 финансовых потоков клуба
q_h8_2 = '''
SELECT 'Аренда ПК' AS source_type, session_date AS event_date, total_paid AS cash_flow
FROM game_sessions
UNION ALL
SELECT 'Бар клуба', order_date, total_amount
FROM bar_orders
UNION ALL
SELECT 'Взнос в турнир', t.start_date, t.entry_fee
FROM tournament_entries te
JOIN tournaments t ON te.tournament_id = t.tournament_id
UNION ALL
SELECT 'Выплата призовых', t.start_date, -te.prize_won
FROM tournament_entries te
JOIN tournaments t ON te.tournament_id = t.tournament_id
WHERE te.prize_won > 0
ORDER BY event_date DESC;
'''
df_audit = pd.read_sql_query(q_h8_2, conn_hard)
print(f"Всего финансовых операций в аудите: {len(df_audit)}")
print(f"Итоговое сальдо клуба по всем операциям: {df_audit['cash_flow'].sum()} руб.")
df_audit.head(15)


Всего финансовых операций в аудите: 1441
Итоговое сальдо клуба по всем операциям: 387195 руб.


source_type,event_date,cash_flow
Взнос в турнир,2026-05-15,350
Взнос в турнир,2026-05-15,350
Взнос в турнир,2026-05-15,350
Взнос в турнир,2026-05-15,350
Взнос в турнир,2026-05-15,350
Взнос в турнир,2026-05-15,350
Взнос в турнир,2026-05-15,350
Взнос в турнир,2026-05-15,350
Взнос в турнир,2026-05-15,350
Взнос в турнир,2026-05-15,350


### Задание 9. Параметризованный срез Net LTV по городам и порогу прибыли
Отбор VIP-клиентов через производную таблицу в `FROM` и параметры `?`.


In [45]:
# 9.1 Параметризованный срез Net LTV
min_profit = 2500
cities = ('Москва', 'Санкт-Петербург')

q_h9_1 = f"""
SELECT 
    sub.member_id,
    sub.nickname,
    sub.city,
    sub.loyalty_tier,
    sub.game_spend,
    sub.bar_spend,
    sub.tournaments_fee,
    sub.prizes_earned,
    sub.net_profit
FROM ({q_net_ltv.rstrip(";\n")}) AS sub
WHERE sub.city IN (?, ?)
  AND sub.net_profit >= ?
ORDER BY sub.net_profit DESC;
"""
df_vip_equity = pd.read_sql_query(q_h9_1, conn_hard, params=(cities[0], cities[1], min_profit))
print(f"Клиентов из Москвы и СПб с прибылью >= {min_profit} руб.: {len(df_vip_equity)}")
df_vip_equity.head(10)


Клиентов из Москвы и СПб с прибылью >= 2500 руб.: 83


member_id,nickname,city,loyalty_tier,game_spend,bar_spend,tournaments_fee,prizes_earned,net_profit
72,Slayer_881,Москва,Gold,2953,8580,0,0,11533
145,Dragon_596,Санкт-Петербург,Silver,4624,6350,0,0,10974
146,CyberKnight_667,Москва,Silver,7370,2960,0,0,10330
96,Neo_347,Москва,Bronze,5412,4360,300,0,10072
73,Frost_65,Санкт-Петербург,Bronze,3276,5945,400,0,9621
108,Titan_769,Москва,Bronze,3300,5980,0,0,9280
150,Shadow_10,Москва,Silver,3894,5070,0,0,8964
135,Falcon_483,Санкт-Петербург,Gold,7493,1470,0,0,8963
103,Frost_890,Москва,Bronze,4250,2550,2050,0,8850
144,Reaper_993,Москва,Bronze,1990,6805,0,0,8795


### Задание 10. Экспорт управленческой витрины Hard
Выгрузка рассчитанной витрины в `executive_customer_equity.csv`.


In [46]:
# Экспорт управленческой витрины Net LTV
export_path_h = base_dir / "executive_customer_equity.csv"
df_net_ltv.to_csv(export_path_h, sep=";", index=False, encoding="utf-8")
print(f"[OK] Управленческая витрина Hard экспортирована: {export_path_h} ({len(df_net_ltv)} строк)")


[OK] Управленческая витрина Hard экспортирована: executive_customer_equity.csv (200 строк)


---
## Критерии оценивания практической работы #7

| Оценка | Требования к выполнению |
| :--- | :--- |
| **«Отлично» (5)** | * Корректно выполнены все части выбранного уровня (включая работу с 4, 5 или 6 таблицами).<br>* **Четкое понимание ловушки `NOT IN + NULL`**: студент осознанно использует `IS NOT NULL` или `NOT EXISTS`.<br>* Правильно составлены коррелированные подзапросы со связкой внешнего и внутреннего контекста.<br>* Использована параметризация запросов с плейсхолдерами `?`.<br>* Для всех уровней успешно сформированы и экспортированы итоговые CSV-витрины данных.<br>* В блокноте отсутствуют ошибки `OperationalError: subquery returns more than 1 row` и декартовы умножения строк. |
| **«Хорошо» (4)** | * Выполнены базовые задачи: скалярные подзапросы, списки `IN`, операции над множествами `UNION` / `INTERSECT`.<br>* Допущены незначительные неточности в оформлении псевдонимов в `FROM` или забыта обработка `IFNULL()` в витрине `SELECT`.<br>* Решены все запросы выбранного уровня, но допущены мелкие помарки в форматировании отчета или комментариях. |
| **«Удовлетворительно» (3)** | * Работа выполнена только на уровне Easy.<br>* Студент не понимает механизма работы `NOT IN` при наличии `NULL` (запрос возвращает 0 строк без понимания причины).<br>* Коррелированные подзапросы не освоены (заменены на громоздкие серии раздельных ячеек в Python).<br>* Не выполнена параметризация через `?` (хардкод значений строковой конкатенацией). |
| **«Неудовлетворительно» (2)** | * Блокнот не запускается из-за синтаксических ошибок в SQL.<br>* Схема базы данных нарушена (не созданы внешние ключи, таблицы не связаны).<br>* Задания списаны или скопированы без запуска кода. |
